# Milestone 2: Shared-Memory Parallelism & Load Balancing

**PDC Project — Point-in-Polygon Classification**

| Task | Owner | Implementation |
|------|-------|----------------|
| Thread management | Saad | OpenMP static + dynamic scheduling |
| Work-stealing load balancing | Fawad | `std::atomic` task pool + OMP dynamic |
| Memory optimisation (spatial tiling) | Fawad | `spatialSort` 32x32 grid |
| SIMD vectorisation (bonus) | Umair | AVX2 batch bbox filter (4 doubles/cycle) |
| Thread-safe spatial index | Ibrahim | Read-only Quadtree + thread-local scratch |
| Correctness verification | All | Brute-force vs indexed on 10 000 pts |

In [ ]:
%%writefile milestone2.cpp
// ============================================================
//  Milestone 2: Shared-Memory Parallelism & Load Balancing
//  Point-in-Polygon Classification — Parallel & Distributed
// ============================================================
//
//  Compile (with AVX2 SIMD bonus):
//    g++ -std=c++17 -O3 -fopenmp -mavx2 -march=native milestone2.cpp -o milestone2
//  Compile (without AVX2, works on all x86):
//    g++ -std=c++17 -O3 -fopenmp milestone2.cpp -o milestone2
//  Run:
//    ./milestone2
//
//  M2 rubric coverage:
//  [SAAD]   Thread management     — OpenMP static + dynamic scheduling
//  [FAWAD]  Load balancing        — Custom work-stealing pool (std::atomic)
//  [FAWAD]  Memory optimisation   — Spatial tiling (spatialSort) for L1/L2
//  [UMAIR]  SIMD bonus            — AVX2 batch bbox filter (4 doubles/cycle)
//  [IBRAHIM]Thread-safe index     — Read-only Quadtree, thread-local scratch
//  [ALL]    Correctness check     — Brute-force vs indexed on 10 000 points
// ============================================================

#include <algorithm>
#include <atomic>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <iomanip>
#include <iostream>
#include <limits>
#include <memory>
#include <numeric>
#include <random>
#include <sstream>
#include <string>
#include <thread>
#include <utility>
#include <vector>
#include <omp.h>

// AVX2 SIMD (bonus): compile with -mavx2 to enable; graceful fallback otherwise
#ifdef __AVX2__
#  include <immintrin.h>
#  define HAS_AVX2 1
#else
#  define HAS_AVX2 0
#endif

namespace pip {

// =============================================================================
// SECTION 1 — Geometry primitives (carried forward from Milestone 1)
// =============================================================================

constexpr double EPS = 1e-9;

struct Point { double x{}, y{}; };

struct BoundingBox {
    double minX{ std::numeric_limits<double>::infinity()};
    double minY{ std::numeric_limits<double>::infinity()};
    double maxX{-std::numeric_limits<double>::infinity()};
    double maxY{-std::numeric_limits<double>::infinity()};

    bool isValid() const { return minX <= maxX && minY <= maxY; }

    void expand(const Point& p) {
        minX = std::min(minX, p.x); minY = std::min(minY, p.y);
        maxX = std::max(maxX, p.x); maxY = std::max(maxY, p.y);
    }
    void expand(const BoundingBox& o) {
        if (!o.isValid()) return;
        minX = std::min(minX, o.minX); minY = std::min(minY, o.minY);
        maxX = std::max(maxX, o.maxX); maxY = std::max(maxY, o.maxY);
    }
    bool contains(const Point& p, double eps = EPS) const {
        return p.x >= minX - eps && p.x <= maxX + eps &&
               p.y >= minY - eps && p.y <= maxY + eps;
    }
    bool contains(const BoundingBox& o, double eps = EPS) const {
        return o.minX >= minX - eps && o.maxX <= maxX + eps &&
               o.minY >= minY - eps && o.maxY <= maxY + eps;
    }
    bool intersects(const BoundingBox& o, double eps = EPS) const {
        return !(o.minX > maxX + eps || o.maxX < minX - eps ||
                 o.minY > maxY + eps || o.maxY < minY - eps);
    }
};

// Data-oriented Polygon: flat vertex buffer + ring offsets.
// ringOffsets layout: [0, outerEnd, hole1End, ..., totalVerts]
// All coordinates contiguous in memory — cache-friendly for parallel access.
struct Polygon {
    int id{};
    std::string name;
    BoundingBox bbox;
    std::vector<Point>    vertices;
    std::vector<uint32_t> ringOffsets;
};

enum class Location { Outside, Boundary, Inside };

inline double cross(const Point& a, const Point& b, const Point& c) {
    return (b.x - a.x) * (c.y - a.y) - (b.y - a.y) * (c.x - a.x);
}

// Scale-independent perpendicular distance check — correct for long thin segments.
inline bool pointOnSegment(const Point& p, const Point& a, const Point& b) {
    if (p.x < std::min(a.x, b.x) - EPS || p.x > std::max(a.x, b.x) + EPS) return false;
    if (p.y < std::min(a.y, b.y) - EPS || p.y > std::max(a.y, b.y) + EPS) return false;
    const double pCross   = std::fabs(cross(a, b, p));
    const double segLenSq = (b.x - a.x)*(b.x - a.x) + (b.y - a.y)*(b.y - a.y);
    return segLenSq < EPS * EPS || (pCross * pCross / segLenSq) <= EPS * EPS;
}

// Ray-casting over a raw pointer ring (no copy; works directly on flat buffer).
Location pointInRing(const Point& p, const Point* pts, size_t n) {
    if (n < 3) return Location::Outside;
    bool inside = false;
    for (size_t i = 0, j = n - 1; i < n; j = i++) {
        if (pointOnSegment(p, pts[j], pts[i])) return Location::Boundary;
        bool aj = (pts[j].y >= p.y), bi = (pts[i].y >= p.y);
        if (aj != bi) {
            double xi = pts[j].x + (p.y - pts[j].y) *
                        (pts[i].x - pts[j].x) / (pts[i].y - pts[j].y);
            if (xi > p.x) inside = !inside;
        }
    }
    return inside ? Location::Inside : Location::Outside;
}

// Full polygon test: outer ring first, then all hole rings.
Location pointInPolygon(const Point& p, const Polygon& poly) {
    if (!poly.bbox.contains(p)) return Location::Outside;
    if (poly.ringOffsets.size() < 2) return Location::Outside;

    const Point* verts = poly.vertices.data();
    Location outer = pointInRing(p, verts + poly.ringOffsets[0],
                                     poly.ringOffsets[1] - poly.ringOffsets[0]);
    if (outer == Location::Outside)  return Location::Outside;
    if (outer == Location::Boundary) return Location::Boundary;

    for (size_t r = 1; r + 1 < poly.ringOffsets.size(); ++r) {
        Location hole = pointInRing(p, verts + poly.ringOffsets[r],
                                       poly.ringOffsets[r + 1] - poly.ringOffsets[r]);
        if (hole == Location::Boundary) return Location::Boundary;
        if (hole == Location::Inside)   return Location::Outside;
    }
    return Location::Inside;
}

void finalizePolygon(Polygon& poly) {
    poly.bbox = BoundingBox{};
    for (const auto& v : poly.vertices) poly.bbox.expand(v);
}

// =============================================================================
// SECTION 2 — Quadtree (read-only after build → inherently thread-safe)
// =============================================================================
// Thread-safety note (Ibrahim): build() is called once before any query.
// All threads share the same immutable tree; zero locking required.
// Each caller provides its own scratch vector, eliminating shared mutable state.

class Quadtree {
public:
    Quadtree(BoundingBox bounds, size_t cap = 8, int maxDepth = 12)
        : root_(std::move(bounds), cap, maxDepth, 0) {}

    ~Quadtree() { root_.clear(); }

    void build(const std::vector<Polygon>& polys) {
        for (const auto& p : polys) root_.insert(&p);
    }

    // Concurrent-safe: read-only traversal. out is caller-owned scratch.
    void query(const Point& p, std::vector<const Polygon*>& out) const {
        out.clear();
        root_.query(p, out);
    }

private:
    struct Node {
        BoundingBox boundary;
        size_t cap; int maxD, depth;
        bool split{false};
        std::vector<const Polygon*> items;
        std::vector<std::unique_ptr<Node>> ch;

        Node(BoundingBox b, size_t c, int md, int d)
            : boundary(b), cap(c), maxD(md), depth(d) {}

        void clear() { items.clear(); ch.clear(); split = false; }

        void subdivide() {
            if (split) return;
            double mx = (boundary.minX + boundary.maxX) * 0.5;
            double my = (boundary.minY + boundary.maxY) * 0.5;
            ch.push_back(std::make_unique<Node>(BoundingBox{boundary.minX, my, mx, boundary.maxY}, cap, maxD, depth+1));
            ch.push_back(std::make_unique<Node>(BoundingBox{mx, my, boundary.maxX, boundary.maxY}, cap, maxD, depth+1));
            ch.push_back(std::make_unique<Node>(BoundingBox{boundary.minX, boundary.minY, mx, my}, cap, maxD, depth+1));
            ch.push_back(std::make_unique<Node>(BoundingBox{mx, boundary.minY, boundary.maxX, my}, cap, maxD, depth+1));
            split = true;
        }

        int fitChild(const BoundingBox& box) const {
            if (!split) return -1;
            for (int i = 0; i < 4; ++i)
                if (ch[i]->boundary.contains(box)) return i;
            return -1;
        }

        void pushDown() {
            if (!split) return;
            std::vector<const Polygon*> stay;
            for (auto* item : items) {
                int idx = fitChild(item->bbox);
                if (idx >= 0) ch[idx]->insert(item);
                else stay.push_back(item);
            }
            items.swap(stay);
        }

        bool insert(const Polygon* poly) {
            if (!boundary.intersects(poly->bbox)) return false;
            if (split) {
                int idx = fitChild(poly->bbox);
                if (idx >= 0) return ch[idx]->insert(poly);
            }
            if (items.size() < cap || depth >= maxD) { items.push_back(poly); return true; }
            if (!split) { subdivide(); pushDown(); }
            int idx = fitChild(poly->bbox);
            if (idx >= 0) return ch[idx]->insert(poly);
            items.push_back(poly);
            return true;
        }

        void query(const Point& p, std::vector<const Polygon*>& out) const {
            if (!boundary.contains(p)) return;
            for (auto* item : items)
                if (item->bbox.contains(p)) out.push_back(item);
            if (!split) return;
            for (const auto& c : ch)
                if (c->boundary.contains(p)) c->query(p, out);
        }
    };
    Node root_;
};

// =============================================================================
// SECTION 3 — Point generation & spatial tiling (cache optimisation — Fawad)
// =============================================================================

std::vector<Point> generateUniform(size_t n, const BoundingBox& d, std::mt19937_64& rng) {
    std::uniform_real_distribution<double> dx(d.minX, d.maxX), dy(d.minY, d.maxY);
    std::vector<Point> pts(n);
    for (auto& p : pts) p = {dx(rng), dy(rng)};
    return pts;
}

std::vector<Point> generateClustered(size_t n, const BoundingBox& d, std::mt19937_64& rng) {
    std::vector<Point> pts;
    pts.reserve(n);
    std::vector<Point> centers = {{25,25},{50,50},{75,70},{35,80}};
    std::discrete_distribution<int> pick({35,30,20,15});
    std::normal_distribution<double> noise(0, 6);
    std::uniform_real_distribution<double> ux(d.minX,d.maxX), uy(d.minY,d.maxY), mix(0,1);
    auto clamp = [](double v, double lo, double hi){ return std::max(lo, std::min(v, hi)); };
    for (size_t i = 0; i < n; ++i) {
        if (mix(rng) < 0.80) {
            auto c = centers[pick(rng)];
            pts.push_back({clamp(c.x + noise(rng), d.minX, d.maxX),
                           clamp(c.y + noise(rng), d.minY, d.maxY)});
        } else {
            pts.push_back({ux(rng), uy(rng)});
        }
    }
    return pts;
}

// Spatial tiling: sort points into a flat 2-D grid cell index so spatially
// adjacent points appear consecutively in memory.  Adjacent points hit the
// same quadtree nodes → warm L1/L2 cache lines → fewer cache misses per query.
void spatialSort(std::vector<Point>& pts, const BoundingBox& d, int res = 32) {
    const double rw = d.maxX - d.minX, rh = d.maxY - d.minY;
    std::sort(pts.begin(), pts.end(), [&](const Point& a, const Point& b) {
        int ax = std::max(0, std::min((int)((a.x - d.minX) / rw * res), res - 1));
        int ay = std::max(0, std::min((int)((a.y - d.minY) / rh * res), res - 1));
        int bx = std::max(0, std::min((int)((b.x - d.minX) / rw * res), res - 1));
        int by = std::max(0, std::min((int)((b.y - d.minY) / rh * res), res - 1));
        return (ay * res + ax) < (by * res + bx);
    });
}

// =============================================================================
// SECTION 4 — Classification helpers
// =============================================================================

// Hot-path classify: avoids heap allocation in the inner loop.
// Dedup via linear scan — candidate count from the quadtree is typically < 20,
// so a linear scan is faster than constructing an unordered_set each call.
inline void classifyPoint(const Point& p,
                           const Quadtree& idx,
                           std::vector<const Polygon*>& scratch,  // thread-local reuse
                           std::vector<int>& out) {
    idx.query(p, scratch);
    out.clear();
    for (const auto* poly : scratch) {
        if (pointInPolygon(p, *poly) != Location::Outside) {
            int id = poly->id;
            bool dup = false;
            for (int m : out) if (m == id) { dup = true; break; }
            if (!dup) out.push_back(id);
        }
    }
    std::sort(out.begin(), out.end());
}

// Brute-force reference — used only for correctness verification, not benchmarked.
std::vector<int> classifyBrute(const Point& p, const std::vector<Polygon>& polys) {
    std::vector<int> out;
    for (const auto& poly : polys)
        if (pointInPolygon(p, poly) != Location::Outside)
            out.push_back(poly.id);
    std::sort(out.begin(), out.end());
    return out;
}

// =============================================================================
// SECTION 5 — SIMD batch bounding-box filter (Umair — AVX2 bonus)
// =============================================================================
// Strategy: the expensive PiP test is only called for candidates that survive
// the bbox filter.  With AVX2 we check 4 polygon bboxes simultaneously using
// 256-bit SIMD registers (__m256d = 4 doubles).
//
// bboxMask4: checks point p against polys[0..count-1]->bbox in one SIMD pass.
// Returns a 4-bit mask: bit i = 1 iff p is within polys[i]->bbox.
// Extra lanes (count < 4) are padded with impossible bboxes → no false hits.

#if HAS_AVX2
inline int bboxMask4(const Point& p, const Polygon* const* polys, int count) {
    alignas(32) double mnX[4] = {1e18,1e18,1e18,1e18};
    alignas(32) double mnY[4] = {1e18,1e18,1e18,1e18};
    alignas(32) double mxX[4] = {-1e18,-1e18,-1e18,-1e18};
    alignas(32) double mxY[4] = {-1e18,-1e18,-1e18,-1e18};
    for (int i = 0; i < count; ++i) {
        mnX[i] = polys[i]->bbox.minX; mnY[i] = polys[i]->bbox.minY;
        mxX[i] = polys[i]->bbox.maxX; mxY[i] = polys[i]->bbox.maxY;
    }
    __m256d px   = _mm256_set1_pd(p.x),      py   = _mm256_set1_pd(p.y);
    __m256d vMnX = _mm256_load_pd(mnX),      vMxX = _mm256_load_pd(mxX);
    __m256d vMnY = _mm256_load_pd(mnY),      vMxY = _mm256_load_pd(mxY);
    __m256d inX  = _mm256_and_pd(_mm256_cmp_pd(px, vMnX, _CMP_GE_OQ),
                                  _mm256_cmp_pd(px, vMxX, _CMP_LE_OQ));
    __m256d inY  = _mm256_and_pd(_mm256_cmp_pd(py, vMnY, _CMP_GE_OQ),
                                  _mm256_cmp_pd(py, vMxY, _CMP_LE_OQ));
    return _mm256_movemask_pd(_mm256_and_pd(inX, inY)) & ((1 << count) - 1);
}

// SIMD-accelerated classify: batch bbox in groups of 4, then scalar PiP only
// for polygons that pass.  Skips entire SIMD lanes that fail the bbox check.
inline void classifyPointSIMD(const Point& p,
                               const Quadtree& idx,
                               std::vector<const Polygon*>& scratch,
                               std::vector<int>& out) {
    idx.query(p, scratch);
    out.clear();
    const int n = (int)scratch.size();
    for (int base = 0; base < n; base += 4) {
        int batch = std::min(4, n - base);
        int mask  = bboxMask4(p, scratch.data() + base, batch);
        while (mask) {
            int lane = __builtin_ctz(mask);   // lowest set bit position
            mask    &= mask - 1;              // clear lowest set bit
            const Polygon* poly = scratch[base + lane];
            if (pointInPolygon(p, *poly) != Location::Outside) {
                int id = poly->id;
                bool dup = false;
                for (int m : out) if (m == id) { dup = true; break; }
                if (!dup) out.push_back(id);
            }
        }
    }
    std::sort(out.begin(), out.end());
}
#endif // HAS_AVX2

// =============================================================================
// SECTION 6 — Work-stealing task pool (Fawad)
// =============================================================================
// Each thread atomically claims the next unclaimed task via fetch_add.
// Effect: fast threads naturally absorb more work; threads processing dense
// polygon clusters fall behind without blocking any other thread.
// This is equivalent to work-stealing in throughput and load-balance outcomes.
// memory_order_relaxed is safe: tasks are independent — no ordering needed.

struct WorkPool {
    std::atomic<size_t> next{0};
    const size_t        total;
    explicit WorkPool(size_t n) : total(n) {}
    size_t claim() { return next.fetch_add(1, std::memory_order_relaxed); }
};

// =============================================================================
// SECTION 7 — Parallel benchmark runners
// =============================================================================

struct RunStats {
    double seconds{};
    size_t totalMatches{};
    std::string label;
};

// -- Serial baseline ----------------------------------------------------------
RunStats runSerial(const std::vector<Point>& pts, const Quadtree& idx) {
    using C = std::chrono::high_resolution_clock;
    std::vector<const Polygon*> scratch;
    std::vector<int> res;
    size_t matches = 0;
    auto t0 = C::now();
    for (const auto& p : pts) {
        classifyPoint(p, idx, scratch, res);
        matches += res.size();
    }
    return {std::chrono::duration<double>(C::now()-t0).count(), matches, "Serial baseline"};
}

// -- OpenMP static scheduling (Saad) -----------------------------------------
// Equal-sized contiguous chunks pre-assigned to threads.
// Best case: uniform data with even polygon density across the domain.
// Weakness: spatial clusters cause some threads to finish far later than others.
RunStats runOMP_Static(const std::vector<Point>& pts, const Quadtree& idx, int threads) {
    using C = std::chrono::high_resolution_clock;
    size_t totalMatches = 0;
    auto t0 = C::now();
    #pragma omp parallel num_threads(threads) reduction(+:totalMatches)
    {
        std::vector<const Polygon*> scratch;   // thread-local: zero contention
        std::vector<int> res;
        #pragma omp for schedule(static)
        for (size_t i = 0; i < pts.size(); ++i) {
            classifyPoint(pts[i], idx, scratch, res);
            totalMatches += res.size();
        }
    }
    return {std::chrono::duration<double>(C::now()-t0).count(), totalMatches,
            "OMP static   T=" + std::to_string(threads)};
}

// -- OpenMP dynamic scheduling (Saad) ----------------------------------------
// Tasks dispatched in chunks of 256 on demand.
// Idle threads immediately pull the next chunk — handles spatial skew well.
RunStats runOMP_Dynamic(const std::vector<Point>& pts, const Quadtree& idx, int threads) {
    using C = std::chrono::high_resolution_clock;
    size_t totalMatches = 0;
    auto t0 = C::now();
    #pragma omp parallel num_threads(threads) reduction(+:totalMatches)
    {
        std::vector<const Polygon*> scratch;
        std::vector<int> res;
        #pragma omp for schedule(dynamic, 256)
        for (size_t i = 0; i < pts.size(); ++i) {
            classifyPoint(pts[i], idx, scratch, res);
            totalMatches += res.size();
        }
    }
    return {std::chrono::duration<double>(C::now()-t0).count(), totalMatches,
            "OMP dynamic  T=" + std::to_string(threads)};
}

// -- Work-stealing pool (Fawad) -----------------------------------------------
// std::thread workers + atomic task counter.  No OMP runtime dependency.
// Threads write to disjoint matchCounts indices — zero data races, no mutex.
RunStats runWorkStealing(const std::vector<Point>& pts, const Quadtree& idx, int threads) {
    using C = std::chrono::high_resolution_clock;
    std::vector<size_t> matchCounts(pts.size(), 0); // one slot per task
    WorkPool pool(pts.size());
    auto t0 = C::now();
    {
        std::vector<std::thread> workers;
        workers.reserve(threads);
        for (int t = 0; t < threads; ++t) {
            workers.emplace_back([&] {
                std::vector<const Polygon*> scratch;
                std::vector<int> res;
                for (;;) {
                    size_t i = pool.claim();
                    if (i >= pool.total) break;
                    classifyPoint(pts[i], idx, scratch, res);
                    matchCounts[i] = res.size();  // no race: unique index per task
                }
            });
        }
        for (auto& w : workers) w.join();
    }
    size_t total = 0;
    for (auto c : matchCounts) total += c;
    return {std::chrono::duration<double>(C::now()-t0).count(), total,
            "WorkStealing T=" + std::to_string(threads)};
}

#if HAS_AVX2
// -- SIMD + OMP (Umair bonus) -------------------------------------------------
RunStats runSIMD_Parallel(const std::vector<Point>& pts, const Quadtree& idx, int threads) {
    using C = std::chrono::high_resolution_clock;
    size_t totalMatches = 0;
    auto t0 = C::now();
    #pragma omp parallel num_threads(threads) reduction(+:totalMatches)
    {
        std::vector<const Polygon*> scratch;
        std::vector<int> res;
        #pragma omp for schedule(dynamic, 256)
        for (size_t i = 0; i < pts.size(); ++i) {
            classifyPointSIMD(pts[i], idx, scratch, res);
            totalMatches += res.size();
        }
    }
    return {std::chrono::duration<double>(C::now()-t0).count(), totalMatches,
            "SIMD+OMP     T=" + std::to_string(threads)};
}
#endif

// =============================================================================
// SECTION 8 — Dataset builder (matches M1 for reproducibility)
// =============================================================================

std::vector<Polygon> buildDataset(int gridX, int gridY) {
    std::vector<Polygon> regions;
    regions.reserve(gridX * gridY);
    int id = 1000;
    const double stepX = 100.0 / gridX, stepY = 100.0 / gridY;
    for (int ix = 0; ix < gridX; ++ix) {
        for (int iy = 0; iy < gridY; ++iy) {
            double x0 = ix * stepX,       y0 = iy * stepY;
            double x1 = x0 + stepX * 0.9, y1 = y0 + stepY * 0.9;
            Polygon poly;
            poly.id   = id++;
            poly.name = "Cell-" + std::to_string(poly.id);
            poly.ringOffsets.push_back(0);
            poly.vertices.push_back({x0,y0}); poly.vertices.push_back({x1,y0});
            poly.vertices.push_back({x1,y1}); poly.vertices.push_back({x0,y1});
            double mx = (x0+x1)*0.5, my = (y0+y1)*0.5;
            double hx = (x1-x0)*0.2, hy = (y1-y0)*0.2;
            poly.ringOffsets.push_back((uint32_t)poly.vertices.size());
            poly.vertices.push_back({mx-hx,my-hy}); poly.vertices.push_back({mx+hx,my-hy});
            poly.vertices.push_back({mx+hx,my+hy}); poly.vertices.push_back({mx-hx,my+hy});
            poly.ringOffsets.push_back((uint32_t)poly.vertices.size());
            finalizePolygon(poly);
            regions.push_back(std::move(poly));
        }
    }
    return regions;
}

// =============================================================================
// SECTION 9 — Correctness verification
// =============================================================================

void verifyCorrectness(const std::vector<Point>& pts,
                       const std::vector<Polygon>& polys,
                       const Quadtree& idx) {
    std::vector<const Polygon*> scratch;
    std::vector<int> indexed;
    size_t errs = 0;
    for (size_t i = 0; i < pts.size() && errs < 5; ++i) {
        classifyPoint(pts[i], idx, scratch, indexed);
        auto brute = classifyBrute(pts[i], polys);
        if (indexed != brute) {
            std::ostringstream oss;
            oss << "Point (" << pts[i].x << "," << pts[i].y
                << "): indexed=" << indexed.size()
                << " brute=" << brute.size();
            std::cerr << "  [MISMATCH] " << oss.str() << "\n";
            ++errs;
        }
    }
    if (errs) throw std::runtime_error("Correctness verification failed.");
}

// =============================================================================
// SECTION 10 — Reporting
// =============================================================================

void printRow(const RunStats& s, double base) {
    std::cout << std::left  << std::setw(22) << s.label
              << " | " << std::fixed << std::setprecision(4) << std::setw(8) << s.seconds << " s"
              << " | speedup " << std::setprecision(2) << std::setw(5) << (base / s.seconds) << "x"
              << " | matches " << s.totalMatches << "\n";
}

void sep(char c = '-', int n = 76) { std::cout << std::string(n, c) << "\n"; }

} // namespace pip

// =============================================================================
// main
// =============================================================================

int main() {
    using namespace pip;
    try {
        std::cout << "=== Milestone 2: Shared-Memory Parallelism & Load Balancing ===\n\n";
        std::cout << "AVX2 SIMD  : " << (HAS_AVX2 ? "ENABLED" : "not available") << "\n";
        std::cout << "OMP threads: " << omp_get_max_threads() << " (max available)\n\n";

        // Build polygon dataset -------------------------------------------------
        const BoundingBox domain{0, 0, 100, 100};
        std::cout << "Building dataset: 50x40 grid = 2000 polygons, each with interior hole...\n";
        auto polys = buildDataset(50, 40);
        Quadtree idx(domain, 8, 12);
        idx.build(polys);
        std::cout << "  " << polys.size() << " polygons indexed.\n\n";

        // Correctness verification ----------------------------------------------
        std::mt19937_64 rng(42);
        std::cout << "Correctness check: quadtree classify vs brute-force on 10 000 points...\n";
        auto verifyPts = generateUniform(10'000, domain, rng);
        verifyCorrectness(verifyPts, polys, idx);
        std::cout << "  [OK] All indexed results agree with brute-force.\n\n";

        // Generate & tile benchmark points -------------------------------------
        constexpr size_t N = 1'000'000;
        std::cout << "Generating " << N << " benchmark points...\n";
        auto uPts = generateUniform(N, domain, rng);
        auto cPts = generateClustered(N, domain, rng);
        spatialSort(uPts, domain);
        spatialSort(cPts, domain);
        std::cout << "  Spatial tiling applied (32x32 grid, L1/L2 cache optimisation).\n\n";

        // Thread counts to benchmark -------------------------------------------
        int maxT = omp_get_max_threads();
        std::vector<int> TC;
        for (int t : {1, 2, 4, 8}) if (t <= maxT) TC.push_back(t);
        {
            bool found = false;
            for (int t : TC) if (t == maxT) { found = true; break; }
            if (!found && maxT > 1) TC.push_back(maxT);
        }

        // Benchmark loop --------------------------------------------------------
        for (const char* name : {"Uniform", "Clustered"}) {
            const auto& pts = (std::string(name) == "Uniform") ? uPts : cPts;
            std::cout << "=== Dataset: " << name << " (" << N << " points) ===\n";
            sep();

            auto serial = runSerial(pts, idx);
            printRow(serial, serial.seconds);
            sep();

            std::cout << "OpenMP static scheduling:\n";
            for (int t : TC) printRow(runOMP_Static(pts, idx, t), serial.seconds);
            sep();

            std::cout << "OpenMP dynamic scheduling (chunk=256):\n";
            for (int t : TC) printRow(runOMP_Dynamic(pts, idx, t), serial.seconds);
            sep();

            std::cout << "Work-stealing (std::thread + atomic task pool):\n";
            for (int t : TC) printRow(runWorkStealing(pts, idx, t), serial.seconds);

#if HAS_AVX2
            sep();
            std::cout << "SIMD AVX2 bbox-filter + OMP dynamic (bonus):\n";
            for (int t : TC) printRow(runSIMD_Parallel(pts, idx, t), serial.seconds);
#endif
            std::cout << "\n";
        }

        // Analysis summary ------------------------------------------------------
        sep('=');
        std::cout <<
            "Design decisions & analysis:\n"
            "  Static scheduling  — even pre-assigned chunks; fast for uniform data\n"
            "                       but suffers load imbalance on clustered GPS data.\n"
            "  Dynamic scheduling — threads self-assign 256-point chunks on demand;\n"
            "                       absorbs spatial skew with low dispatch overhead.\n"
            "  Work-stealing pool — std::atomic fetch_add (relaxed); zero blocking,\n"
            "                       no OMP runtime; load balance identical to dynamic.\n"
            "  Spatial tiling     — spatialSort groups nearby points consecutively;\n"
            "                       warms L1/L2 cache lines for quadtree traversal.\n"
            "  Dedup strategy     — linear scan over <20 candidates avoids heap\n"
            "                       allocation that unordered_set would incur per call.\n";
#if HAS_AVX2
        std::cout <<
            "  AVX2 SIMD          — bboxMask4 tests 4 polygon bboxes per instruction;\n"
            "                       only survivors proceed to scalar ray-casting PiP.\n";
#endif
        sep('=');
        std::cout << "\nMilestone 2 complete.\n";

    } catch (const std::exception& ex) {
        std::cerr << "ERROR: " << ex.what() << "\n";
        return 1;
    }
    return 0;
}


## Compilation

Preferred command enables AVX2 SIMD (bonus).  
Fallback compiles without SIMD and works on any x86-64 machine.

In [ ]:
# With AVX2 SIMD (preferred)
!g++ -std=c++17 -O3 -fopenmp -mavx2 -march=native milestone2.cpp -o milestone2 && echo "Compiled OK (AVX2 enabled)"

In [ ]:
# Fallback without AVX2
# !g++ -std=c++17 -O3 -fopenmp milestone2.cpp -o milestone2 && echo "Compiled OK (scalar)"

## Run

In [ ]:
!./milestone2